[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HunterBushnell/SCP/blob/main/0_pipeline.ipynb)

# SCP Pipeline - Compact Steps 1-5

This is the simplest end-to-end SCP notebook for an already staged cell. It fills missing configs safely, builds one cell for passive/active/synapse tuning, then runs the final simulation in a clean process and displays its result.

Use `1_setup.ipynb` for full model import/setup control. Use the numbered notebooks when you need ACT active tuning, target-source setup, exports, synapse optimization, or other advanced options.

> **Manual tuning rule:** after editing a fit JSON, HOC/morphology source, `cell_config.json`, or MOD source, restart the kernel and rerun from the top. Runtime, target, geometry, and synapse config edits are reloaded where they are used.

## Notebook Setup: Environment

Run this once. The cell finds SCP locally or clones it in a fresh Colab runtime. Its source is collapsed to keep the workflow focused.

In [ ]:
try:
    ip = get_ipython()
except NameError:
    ip = None
if ip is not None:
    try:
        ip.run_line_magic("load_ext", "autoreload")
        ip.run_line_magic("autoreload", "2")
    except Exception as exc:
        print(f"IPython autoreload unavailable ({exc}); continuing.")

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
SCP_REPO_URL = os.environ.get("SCP_REPO_URL", "https://github.com/HunterBushnell/SCP.git")
SCP_REPO_BRANCH = os.environ.get("SCP_REPO_BRANCH", "") or None
SCP_REPO_DIR = Path(os.environ.get("SCP_REPO_DIR", "/content/SCP" if IN_COLAB else str(Path.cwd() / "SCP")))
INSTALL_DEPS = None  # None installs automatically in Colab and not locally.

def _looks_like_scp(path):
    return (path / "modules").is_dir() and (path / "run_pipeline.py").is_file()

repo_root = None
env_root = os.environ.get("SCP_ROOT")
if env_root and _looks_like_scp(Path(env_root).expanduser()):
    repo_root = Path(env_root).expanduser().resolve()
else:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    for base in (start, start.parent):
        try:
            candidates.extend(child for child in base.iterdir() if child.is_dir())
        except Exception:
            pass
    for candidate in candidates:
        if _looks_like_scp(candidate):
            repo_root = candidate.resolve()
            break

if repo_root is None:
    if not IN_COLAB and os.environ.get("SCP_AUTO_CLONE", "0") not in {"1", "true", "True"}:
        raise FileNotFoundError("Could not find SCP. Set SCP_ROOT or run from the repo.")
    clone_url = SCP_REPO_URL
    token = os.environ.get("SCP_GIT_TOKEN") or os.environ.get("SCP_GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN")
    if token and clone_url.startswith("https://") and "@" not in clone_url:
        clone_url = clone_url.replace("https://", f"https://{token}@", 1)
    clone_command = ["git", "clone", "--depth", "1"]
    if SCP_REPO_BRANCH:
        clone_command += ["--branch", SCP_REPO_BRANCH]
    clone_command += [clone_url, str(SCP_REPO_DIR)]
    subprocess.check_call(clone_command)
    repo_root = SCP_REPO_DIR.resolve()

os.environ["SCP_ROOT"] = str(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from modules.notebooks.bootstrap import finish_step5_notebook_setup, ensure_python_package
setup = finish_step5_notebook_setup(repo_root, install_deps=INSTALL_DEPS, check_external_inputs=False)
repo_root = setup["repo_root"]
IN_COLAB = setup["in_colab"]
if INSTALL_DEPS is True or (INSTALL_DEPS is None and IN_COLAB):
    ensure_python_package("ipywidgets")

from modules.notebooks import (
    prepare_interactive_synapse_tuner,
    prepare_pipeline_notebook,
    run_active_stage,
    run_fresh_simulation,
    run_passive_stage,
    show_run_diagnostics,
)

## Notebook Setup: Settings

For a prepared tune, usually change only `cell_name` and `tune_name`. Set `adb_specimen_id` only when this notebook should download/setup that Allen cell in the selected tune directory.

In [ ]:
# Cell/tune selection
cell_name = "PV"
tune_name = "tuned"
tune_dir_override = None  # Optional path; overrides cell_name/tune_name.

# Optional ADB convenience setup. Leave None for an existing tune.
adb_specimen_id = None
adb_model_type = "perisomatic"
recompile_modfiles = False

# Step 2: passive tuning
passive_amps_pA = [-50, -100]
compute_act_passive_proposal = False  # Review-only; never auto-applied.
passive_protocol_overrides = {}  # e.g. {"stim_dur": 1000, "h_tstop": 1500}

# Step 3: active tuning and FI curve
active_amps_pA = [150, 300]
fi_amps_pA = list(range(0, 301, 50))
active_protocol_overrides = {}

# Step 4: BMTool is optional and disabled for the default Run All path.
enable_synapse_tuning = False
synapse_connection = None  # None uses synapse_tuning_config.json default.

# Step 5: every final run is saved and loaded back for diagnostics.
n_trials = 1
seed = None
run_iclamp = False
output_stem = None  # None -> unique pipeline_<timestamp> folder.
diagnostic_plot = "standard"  # "summary", "standard", "single_plot", or None

## Step 1: Setup and Load the Tune

This safely fills missing configs, compiles configured mechanisms, validates the tune without constructing a throwaway cell, and then builds the one cell reused by Steps 2–4.

In [ ]:
pipeline_state = prepare_pipeline_notebook(
    repo_root=repo_root,
    cell_name=cell_name,
    tune_name=tune_name,
    tune_dir_override=tune_dir_override,
    adb_specimen_id=adb_specimen_id,
    adb_model_type=adb_model_type,
    recompile_modfiles=recompile_modfiles,
)
tune_dir = pipeline_state.tune_dir
cell = pipeline_state.cell

## Step 2: Passive Tuning

Rerun this cell while reviewing passive behavior. It reloads `target_config.json` and simulation conditions. Enabling the ACT proposal requires complete Rin, tau, and resting-voltage targets; proposed values are printed but never written.

In [ ]:
passive_result = run_passive_stage(
    pipeline_state,
    amps_pA=passive_amps_pA,
    compute_act_proposal=compute_act_passive_proposal,
    protocol_overrides=passive_protocol_overrides,
)

## Step 3: Active Tuning and FI Curve

Run the selected positive-current sweeps and FI curve. Targets already configured as FI arrays or CSV are overlaid and compared automatically.

In [ ]:
active_result = run_active_stage(
    pipeline_state,
    active_amps_pA=active_amps_pA,
    fi_amps_pA=fi_amps_pA,
    protocol_overrides=active_protocol_overrides,
)

## Step 4: Synapse Tuning (Optional)

Set `enable_synapse_tuning = True` in Settings to initialize BMTool around the same cell. This compact workflow includes only the single-event and interactive tuner actions. Manually copy chosen values into the relevant synapse JSON before Step 5.

In [ ]:
if enable_synapse_tuning:
    tuner = prepare_interactive_synapse_tuner(
        pipeline_state,
        connection_override=synapse_connection,
    )
else:
    tuner = None
    print("Synapse tuning disabled; using the configured synapse JSONs for Step 5.")

In [ ]:
if tuner is not None:
    tuner.SingleEvent()
else:
    print("SingleEvent skipped.")

In [ ]:
if tuner is not None:
    tuner.InteractiveTuner()
else:
    print("InteractiveTuner skipped.")

## Step 5: Simulate and Diagnose

The final run starts in a fresh Python process, reloads the latest configs, saves to a unique run folder, then loads that result back here. This keeps BMTool and tuning-protocol objects out of the final simulation.

In [ ]:
simulation_result = run_fresh_simulation(
    pipeline_state,
    n_trials=n_trials,
    seed=seed,
    iclamp=run_iclamp,
    output_stem=output_stem,
)
results = simulation_result.results
diagnostics = show_run_diagnostics(
    results,
    diagnostic_plot=diagnostic_plot,
    include_inputs=not run_iclamp,
    cell_name=pipeline_state.context.cell_name,
    tune_name=pipeline_state.context.tune_name,
    repo_root=repo_root,
)
print("Saved run manifest:", simulation_result.manifest_path)

## Next: Step 6 Analysis

Open `6_analysis.ipynb` for full saved-run analysis. Return to the corresponding numbered notebook whenever you need the advanced controls intentionally omitted here.